In [ ]:

pip install sentence-transformers faiss-cpu transformers

In [ ]:
passages = [
    "Machine learning is a field of artificial intelligence that focuses on training algorithms.",
    "Deep learning is a subset of machine learning using neural networks with many layers.",
    "FAISS is a library developed by Facebook for efficient similarity search.",
    "Chroma is a vector database designed for AI applications.",
    "Embeddings are numerical representations of text used in semantic search.",
    "Natural language processing enables computers to understand human language.",
    "Transformers are models that process sequential data using attention mechanisms.",
    "Sentence Transformers generate embeddings for sentences and paragraphs.",
    "Similarity search finds items that are close in vector space.",
    "Vector databases store and retrieve embeddings efficiently."
]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(passages)
print(embeddings.shape)

In [ ]:
import faiss
import numpy as np

# Convert to float32
embeddings = np.array(embeddings).astype("float32")

# Create index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Query
query = input("Enter your query:")
query_embedding = model.encode([query]).astype("float32")

# Search
k = 3
distances, indices = index.search(query_embedding, k)

print("\nTop results:")
for i in indices[0]:
    print("-", passages[i])

In [ ]:
DAY-2

In [ ]:
# Install
!pip -q install pypdf sentence-transformers faiss-cpu nltk

import re
import faiss
import nltk
from pypdf import PdfReader
from google.colab import files
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize

# Download NLTK data
nltk.download("punkt")
nltk.download("punkt_tab")

# 1. Upload PDF
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

# 2. Read PDF text
reader = PdfReader(pdf_file)
pages = [page.extract_text() for page in reader.pages if page.extract_text()]
text = "\n".join(pages)

# 3. Clean text
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)   # remove extra spaces
    return text.strip()

text = clean_text(text)

# 4. Sentence-based chunking
def chunk_text(text, chunk_size=300):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent.split())

        if current_len + sent_len <= chunk_size:
            current_chunk.append(sent)
            current_len += sent_len
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_len = sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

chunks = chunk_text(text, chunk_size=300)
print("Total chunks:", len(chunks))

# 5. Create embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)

# 6. Store in FAISS
index = faiss.IndexFlatIP(embeddings.shape[1])   # cosine similarity
index.add(embeddings)

# 7. Retrieve function
def retrieve(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0]]

# 8. Example

while True:
  query = input("\nEnter your question (type 'exit' to stop): ")
  if query.lower() == "exit":
    print("Exiting...")
    break
  results = retrieve(query, top_k=3)
  for i, r in enumerate(results, 1):
    print(f"\nPassage {i}:\n{r}\n")


In [ ]:
DAY-3

In [ ]:
!pip install -q -U transformers sentence-transformers faiss-cpu accelerate sentencepiece

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
import faiss
import torch

# 1. Small knowledge base
docs = [
  "Python was created by Guido van Rossum.",
  "The chemical formula of water is H2O.",
  "Mount Everest is the highest mountain above sea level.",
  "The human heart has four chambers.",
  "The blue whale is the largest animal on Earth.",
  "The capital of France is Paris.",
  "The Sun is a star at the center of the Solar System.",
  "The Moon is Earth's natural satellite.",
  "The Great Wall is in China.",
  "Bats are mammals, not birds."
]

# 2. Embed documents
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(
    docs,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# 3. Store in FAISS
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

# 4. Retriever
def retrieve(query, top_k=2):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [docs[i] for i in indices[0]]

# 5. Load FLAN-T5 properly
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 6. RAG function
def answer_question(question):
    passages = retrieve(question, top_k=2)
    context = " ".join(passages)

    prompt = f"""Answer using only the context.

Context: {context}
Question: {question}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=30
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# 7. Test
questions = [
    "Who created Python?",
    "What is the formula of water?",
    "How many chambers does the heart have?",
    "What is the highest mountain?",
    "What is the largest animal on Earth?"
]

for q in questions:
    print("Q:", q)
    print("A:", answer_question(q))
    print()

In [ ]:
DAY-4

In [ ]:
!pip install -q -U sentence-transformers transformers rank-bm25

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi
import numpy as np
import torch
import re
# Knowledge base
docs = [
    "Python was created by Guido van Rossum.",
    "The chemical formula of water is H2O.",
    "Mount Everest is the highest mountain above sea level.",
    "The human heart has four chambers.",
    "The blue whale is the largest animal on Earth.",
    "The capital of France is Paris.",
    "The Sun is a star at the center of the Solar System.",
    "The Moon is Earth's natural satellite.",
    "The Great Wall is in China.",
    "Bats are mammals, not birds."
]
# Models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)
# Embeddings
doc_emb = embed_model.encode(
    docs,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")
# BM25
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_docs = [tokenize(doc) for doc in docs]
bm25 = BM25Okapi(tokenized_docs)
# Hybrid retrieval using RAW scores
# hybrid = alpha*vector_raw + beta*bm25_raw
def retrieve_hybrid(query, top_k=3, alpha=0.7, beta=0.3):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    vec_scores = np.dot(doc_emb, q_emb)   # raw vector scores
    bm25_scores = np.array(bm25.get_scores(tokenize(query)), dtype=np.float32)  # raw bm25 scores

    results = []
    for i, doc in enumerate(docs):
        hybrid_score = alpha * float(vec_scores[i]) + beta * float(bm25_scores[i])
        results.append({
            "doc": doc,
            "vector_score": float(vec_scores[i]),
            "bm25_score": float(bm25_scores[i]),
            "hybrid_score": float(hybrid_score)
        })

    results.sort(key=lambda x: x["hybrid_score"], reverse=True)
    return results[:top_k]
# Answer generation
def generate_answer(question, passages):
    context = " ".join(passages)
    prompt = f"""Answer using only the context.
If the answer is not in the context, say: I don't know.

Context: {context}
Question: {question}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = llm.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# Run
def ask(question, top_k=3):
    print("=" * 100)
    print("QUESTION:", question)
    print("\n--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---")

    results = retrieve_hybrid(question, top_k=top_k)

    for r in results:
        print(f"Doc: {r['doc']}")
        print(f"Vector Score: {r['vector_score']:.4f}")
        print(f"BM25 Score  : {r['bm25_score']:.4f}")
        print(f"Hybrid Score: {r['hybrid_score']:.4f}\n")

    answer = generate_answer(question, [r["doc"] for r in results])
    print("Final Answer:", answer)
# Test
questions = [
    "Who created Python?",
    "What is the formula of water?",
    "Which animal is the largest on Earth?",
    "What is the highest mountain above sea level?",
    "What is the capital of France?"
]

for q in questions:
    ask(q)